# 10-Band DSP Audio Equalizer — Clean Notebook

This notebook focuses only on the important DSP parts of the project:

1. Smart presets
2. Filter design: FIR Kaiser, IIR Butterworth SOS, and RBJ Parametric EQ
3. Before/after audio playback
4. FFT spectrum before and after equalization
5. Clipping warning and automatic normalization check
6. Filter frequency response plot
7. Final method comparison

**Recommended main method for the Streamlit app:** RBJ Parametric EQ.

## 1. Imports and Settings

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Audio, display

from dsp_equalizer import (
    DEFAULT_FS,
    BAND_LABELS,
    PRESETS,
    PRESET_DESCRIPTIONS,
    generate_demo_audio,
    process_audio,
    composite_response,
    fft_spectrum,
    compare_methods,
)

fs = DEFAULT_FS
method = "RBJ Parametric EQ"
preset_name = "Smiley / Music"
gains_db = np.array(PRESETS[preset_name], dtype=float)
fir_taps = 1025

print("Selected method:", method)
print("Selected smart preset:", preset_name)
print("Preset description:", PRESET_DESCRIPTIONS[preset_name])
print("Sample rate:", fs)

pd.DataFrame({"Band (Hz)": BAND_LABELS, "Gain (dB)": gains_db})

## 2. Generate Demo Audio

A synthetic signal is used so the notebook can run without any external audio file.

In [ ]:
x = generate_demo_audio(fs=fs, duration=6.0)
t = np.arange(len(x)) / fs

print("Audio duration:", round(len(x) / fs, 2), "seconds")
print("Number of samples:", len(x))

print("Before audio:")
display(Audio(x, rate=fs))

## 3. Input Waveform

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(t, x, linewidth=0.8)
plt.title("Input Audio Waveform")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.grid(True, alpha=0.25)
plt.xlim(0, t[-1])
plt.show()

## 4. FFT Spectrum Before Equalization

FFT analysis converts audio from the time domain to the frequency domain, so we can see the bass, mid, and treble content.

In [ ]:
freqs_in, mag_in = fft_spectrum(x, fs)

plt.figure(figsize=(12, 4.5))
plt.semilogx(freqs_in[1:], mag_in[1:], linewidth=1)
plt.title("Input Audio FFT Spectrum")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Magnitude (dB)")
plt.xlim(20, 20000)
plt.ylim(np.max(mag_in) - 100, np.max(mag_in) + 5)
plt.grid(True, which="both", alpha=0.25)
plt.show()

## 5. Filter Design: Frequency Response

This plot shows how FIR, IIR, and RBJ respond to the selected smart preset.

In [ ]:
methods = ["FIR Kaiser", "IIR Butterworth SOS", "RBJ Parametric EQ"]

plt.figure(figsize=(12, 5))
for m in methods:
    response = composite_response(m, fs, gains_db, fir_taps=fir_taps)
    lw = 2.8 if m == method else 1.3
    alpha = 1.0 if m == method else 0.55
    plt.semilogx(response["freqs"], response["mag_db"], linewidth=lw, alpha=alpha, label=m)

plt.semilogx(response["freqs"], response["target_db"], linestyle="--", linewidth=2.2, label="Target Slider Curve")
plt.title("Filter Design Frequency Response")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Gain (dB)")
plt.xlim(20, 20000)
plt.ylim(-15, 15)
plt.grid(True, which="both", alpha=0.25)
plt.legend()
plt.show()

## 6. Apply RBJ Parametric EQ

RBJ Parametric EQ is used as the main method because it is fast, low-latency, and works well with interactive sliders.

In [ ]:
y, metrics = process_audio(x, fs, gains_db, method, fir_taps=fir_taps)
t_out = np.arange(len(y)) / fs

metrics_df = pd.DataFrame([metrics])
metrics_df.T

## 7. Clipping Warning and Auto-Normalization

In [ ]:
if metrics["clipping_detected"]:
    print("Warning: clipping risk detected. The processed signal exceeded ±1 before normalization.")
elif metrics["normalization_applied"]:
    print("Info: auto-normalization was applied because the output was above the safe peak level of 0.95.")
else:
    print("No clipping risk detected. The output signal is already safe.")

print("Input peak:", round(metrics["input_peak"], 4))
print("Peak before normalization:", round(metrics["max_abs_before_normalization"], 4))
print("Peak after normalization:", round(metrics["max_abs_after_normalization"], 4))
print("Normalization scale factor:", round(metrics["normalization_scale_factor"], 4))

## 8. Before/After Audio Player

In [ ]:
print("Before audio:")
display(Audio(x, rate=fs))

print("After audio:")
display(Audio(y, rate=fs))

## 9. Output Waveform After Equalization

In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(t_out, y, linewidth=0.8)
plt.title("Output Audio Waveform After Equalization")
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")
plt.grid(True, alpha=0.25)
plt.xlim(0, t_out[-1])
plt.show()

## 10. FFT Spectrum Before vs After Equalization

This is the key proof that the equalizer changed the frequency content of the audio.

In [ ]:
freqs_out, mag_out = fft_spectrum(y, fs)
top = max(np.max(mag_in), np.max(mag_out))

plt.figure(figsize=(12, 5))
plt.semilogx(freqs_in[1:], mag_in[1:], linewidth=1.2, label="Before equalization")
plt.semilogx(freqs_out[1:], mag_out[1:], linewidth=1.2, label="After equalization")
plt.title("FFT Spectrum Before vs After Equalization")
plt.xlabel("Frequency (Hz)")
plt.ylabel("Magnitude (dB)")
plt.xlim(20, 20000)
plt.ylim(top - 100, top + 5)
plt.grid(True, which="both", alpha=0.25)
plt.legend()
plt.show()

## 11. Method Comparison Table

In [ ]:
comparison_df = pd.DataFrame(compare_methods(fs, gains_db, fir_taps=fir_taps))
comparison_df

## 12. Final Recommendation

Use **RBJ Parametric EQ** as the main method in the Streamlit app because it has almost zero latency, low computation cost, and smooth interaction with sliders.

Use **IIR Butterworth SOS** as the strongest technical comparison method because it is stable and efficient.

Keep **FIR Kaiser** for academic explanation because it has linear phase, but it is less suitable for a real-time interactive app because it has higher latency.